# LeetCode #210 — Course Schedule II: tìm một thứ tự tô-pô

**Mức độ:** Medium · **Chủ đề:** Depth-First Search, Breadth-First Search, Graph, Topological Sort

Skeleton trên LeetCode:

```python
class Solution:
    def findOrder(self, numCourses: int, prerequisites: List[List[int]]) -> List[int]:
```

Mục tiêu của notebook:

1. Nối tiếp #207: không chỉ trả lời "học hết được không" mà **in ra một thứ tự học**.
2. Kahn: thứ tự lấy ra khỏi hàng đợi **chính là** thứ tự tô-pô.
3. DFS: thứ tự **hậu tố** (đỉnh xong sau mọi đỉnh con) **đảo ngược** là thứ tự tô-pô — và vì sao.
4. Viết hàm **kiểm tra** một thứ tự có hợp lệ không (vì đáp án không duy nhất).

# 1. Bài toán là gì?

Có `numCourses` môn (`0 .. numCourses − 1`), `prerequisites[i] = [a, b]` nghĩa là **phải học `b` trước `a`**. Trả về **một** thứ tự học hết mọi môn; nếu không thể, trả về `[]`. Có nhiều đáp án đúng thì trả về bất kỳ.

**Constraints (theo đề):**

- `1 <= numCourses <= 2000`
- `0 <= prerequisites.length <= numCourses * (numCourses − 1)`
- `prerequisites[i].length == 2`, `0 <= a, b < numCourses`, `a != b`
- Các cặp **khác nhau**.

---

## Ví dụ

```text
numCourses = 2, prerequisites = [[1, 0]]                          ->  [0, 1]
numCourses = 4, prerequisites = [[1, 0], [2, 0], [3, 1], [3, 2]]  ->  [0, 1, 2, 3] hoặc [0, 2, 1, 3]
numCourses = 1, prerequisites = []                                ->  [0]
numCourses = 2, prerequisites = [[1, 0], [0, 1]]                  ->  []
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Đồ thị (graph) và cách biểu diễn
2. Duyệt theo chiều sâu (DFS — Depth-First Search)
3. Sắp xếp tô-pô (topological sort) và phát hiện chu trình

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Đồ thị (graph) và cách biểu diễn

### Đồ thị là gì?

Đồ thị gồm các **đỉnh** (vertex/node) và các **cạnh** (edge) nối cặp đỉnh. Cạnh có thể **vô hướng** (đi được hai chiều, như đường hai chiều) hoặc **có hướng**. Cây là trường hợp riêng: đồ thị liên thông, không chu trình.

```text
1 ── 2
│    │       đỉnh: 1, 2, 3, 4
4 ── 3       cạnh: (1,2) (2,3) (3,4) (4,1)  → có chu trình 1-2-3-4-1
```

### Ba cách biểu diễn

| Cách | Mô tả | Hợp khi |
|---|---|---|
| Danh sách cạnh | `[(1,2), (2,3), …]` | đầu vào thô |
| **Danh sách kề** | `{1: [2, 4], 2: [1, 3], …}` | hầu hết bài duyệt (BFS/DFS) |
| Ma trận kề | `M[u][v] = 1` nếu có cạnh | đồ thị dày, cần hỏi "có cạnh u-v?" nhanh |
| Đối tượng nút | `node.neighbors = [...]` | LeetCode kiểu "Clone Graph" |

In [1]:
from collections import defaultdict, deque

canh = [(1, 2), (2, 3), (3, 4), (4, 1)]
ke = defaultdict(list)
for u, v in canh:                # vô hướng: thêm cả hai chiều
    ke[u].append(v)
    ke[v].append(u)
print("danh sách kề:", dict(ke))

n = 4
M = [[0] * (n + 1) for _ in range(n + 1)]
for u, v in canh:
    M[u][v] = M[v][u] = 1
print("ma trận kề (bỏ hàng/cột 0):")
for row in M[1:]:
    print("  ", row[1:])

danh sách kề: {1: [2, 4], 2: [1, 3], 3: [2, 4], 4: [3, 1]}
ma trận kề (bỏ hàng/cột 0):
   [0, 1, 0, 1]
   [1, 0, 1, 0]
   [0, 1, 0, 1]
   [1, 0, 1, 0]


### Đồ thị dạng đối tượng nút

Mỗi nút giữ danh sách **tham chiếu** tới các nút hàng xóm. Vì có chu trình, khi duyệt **bắt buộc** phải nhớ nút đã thăm — nếu không sẽ đi vòng mãi.

In [2]:
class GNode:
    def __init__(self, val=0, neighbors=None):
        self.val = val
        self.neighbors = neighbors if neighbors is not None else []

nodes = {i: GNode(i) for i in range(1, 5)}
for u, v in canh:
    nodes[u].neighbors.append(nodes[v])
    nodes[v].neighbors.append(nodes[u])

def duyet_nut(start):
    seen, q, out = {start}, deque([start]), []
    while q:
        x = q.popleft()
        out.append(x.val)
        for y in x.neighbors:
            if y not in seen:
                seen.add(y); q.append(y)
    return out

print("BFS từ nút 1:", duyet_nut(nodes[1]))
print("hàng xóm của 3:", [y.val for y in nodes[3].neighbors])

BFS từ nút 1: [1, 2, 4, 3]
hàng xóm của 3: [2, 4]


### Sao chép **nông** và sao chép **sâu**

- Sao chép nông: tạo đối tượng mới nhưng vẫn **trỏ tới** các nút cũ.
- Sao chép sâu: tạo **mọi** nút mới, và cạnh mới nối các nút **mới** với nhau; không chia sẻ gì với bản gốc.

Với đồ thị có chu trình, sao chép sâu cần một **bảng `cũ → mới`** để mỗi nút cũ chỉ được tạo bản sao **một lần** và các cạnh trỏ đúng bản sao.

In [3]:
import copy
a_list = [[1, 2], [3]]
nong = list(a_list)
sau = copy.deepcopy(a_list)
a_list[0].append(99)
print("gốc:", a_list, "| nông (bị ảnh hưởng):", nong, "| sâu:", sau)

gốc: [[1, 2, 99], [3]] | nông (bị ảnh hưởng): [[1, 2, 99], [3]] | sâu: [[1, 2], [3]]


## 2.2 Duyệt theo chiều sâu (DFS — Depth-First Search)

### DFS là gì?

DFS đi **sâu nhất có thể** theo một nhánh, tới khi hết đường thì **quay lui** về ngã rẽ gần nhất và thử nhánh khác — giống đi mê cung bằng cách luôn bám một bức tường.

```text
        A
       / \          DFS từ A (thử hàng xóm theo thứ tự):
      B   C         A → B → D (cụt, quay lui) → E → F → C
     / \   \
    D   E - F
```

Công cụ: **đệ quy** (dùng call stack) hoặc **ngăn xếp** tự quản lý.

### Khung DFS

```text
def dfs(u):
    visited.add(u)
    xử lý u
    for v in hàng_xóm(u):
        if v not in visited:
            dfs(v)
```

In [4]:
do_thi_dfs = {
    "A": ["B", "C"], "B": ["A", "D", "E"], "C": ["A", "F"],
    "D": ["B"], "E": ["B", "F"], "F": ["C", "E"],
}

def dfs_de_quy(graph, u, visited, thu_tu, sau=0):
    visited.add(u)
    thu_tu.append(u)
    print("    " * sau + f"vào {u}")
    for v in graph[u]:
        if v not in visited:
            dfs_de_quy(graph, v, visited, thu_tu, sau + 1)
    print("    " * sau + f"rời {u}")

tt = []
dfs_de_quy(do_thi_dfs, "A", set(), tt)
print("Thứ tự DFS:", tt)

vào A
    vào B
        vào D
        rời D
        vào E
            vào F
                vào C
                rời C
            rời F
        rời E
    rời B
rời A
Thứ tự DFS: ['A', 'B', 'D', 'E', 'F', 'C']


### DFS bằng ngăn xếp (không đệ quy)

Thay hàng đợi của BFS bằng **ngăn xếp**. Tránh được giới hạn độ sâu đệ quy khi dữ liệu lớn.

In [5]:
def dfs_ngan_xep(graph, start):
    st, visited, thu_tu = [start], set(), []
    while st:
        u = st.pop()
        if u in visited:
            continue
        visited.add(u)
        thu_tu.append(u)
        for v in reversed(graph[u]):        # đảo để hàng xóm đầu tiên được lấy ra trước
            if v not in visited:
                st.append(v)
    return thu_tu

print("DFS bằng ngăn xếp:", dfs_ngan_xep(do_thi_dfs, "A"))

DFS bằng ngăn xếp: ['A', 'B', 'D', 'E', 'F', 'C']


### DFS trên lưới: tô loang (flood fill) và đếm vùng

Lưới là đồ thị ẩn: mỗi ô nối với 4 ô kề. DFS từ một ô sẽ "loang" khắp vùng liên thông chứa nó. Đếm số lần phải bắt đầu DFS mới = số vùng.

In [6]:
def dem_dao(grid):
    g = [list(row) for row in grid]
    R, C = len(g), len(g[0])
    def loang(r, c):
        if r < 0 or r >= R or c < 0 or c >= C or g[r][c] != "1":
            return                              # ra ngoài / nước / đã thăm
        g[r][c] = "#"                           # đánh dấu đã thăm
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            loang(r + dr, c + dc)
    dem = 0
    for r in range(R):
        for c in range(C):
            if g[r][c] == "1":
                dem += 1
                loang(r, c)
    return dem, ["".join(row) for row in g]

print(dem_dao(["11000",
               "11010",
               "00011"]))

(2, ['##000', '##0#0', '000##'])


### DFS trên cây: mang thông tin dọc đường đi

Trên cây không cần `visited` (không có chu trình). DFS hay **truyền xuống** thông tin của đường đi từ gốc (tổng, đường đi hiện tại) và **trả lên** kết quả của cây con.

In [7]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

cay_dfs = TreeNode(5, TreeNode(4, TreeNode(11, TreeNode(7), TreeNode(2))), TreeNode(8, TreeNode(13), TreeNode(4)))

def moi_duong_goc_la(node, duong, out):
    if node is None:
        return
    duong.append(node.val)                     # đi xuống: thêm nút
    if node.left is None and node.right is None:
        out.append((list(duong), sum(duong)))  # tới lá: ghi lại một bản sao
    moi_duong_goc_la(node.left, duong, out)
    moi_duong_goc_la(node.right, duong, out)
    duong.pop()                                # quay lên: bỏ nút

ket_qua = []
moi_duong_goc_la(cay_dfs, [], ket_qua)
for d, s in ket_qua:
    print(d, "tổng =", s)

[5, 4, 11, 7] tổng = 27
[5, 4, 11, 2] tổng = 22
[5, 8, 13] tổng = 26
[5, 8, 4] tổng = 17


### So sánh BFS và DFS

| | BFS | DFS |
|---|---|---|
| Cấu trúc | hàng đợi | ngăn xếp / đệ quy |
| Thứ tự | theo lớp (gần → xa) | theo nhánh (sâu trước) |
| Đường ngắn nhất (không trọng số) | **có** | không đảm bảo |
| Bộ nhớ | độ rộng lớn nhất | độ sâu lớn nhất |
| Hợp với | khoảng cách, theo tầng | liệt kê đường đi, vùng liên thông, quay lui |

Lỗi hay gặp: quên `visited` trên đồ thị có chu trình (lặp vô hạn); đệ quy quá sâu trên lưới lớn; ghi `out.append(duong)` thay vì bản sao `list(duong)` (về sau list bị sửa).

## 2.3 Sắp xếp tô-pô (topological sort) và phát hiện chu trình

### Bài toán

Cho đồ thị **có hướng**, cạnh `u → v` nghĩa là "`u` phải làm trước `v`". **Thứ tự tô-pô** là cách xếp mọi đỉnh thành dãy sao cho mọi cạnh đi từ **trước ra sau**. Tồn tại ⟺ đồ thị **không có chu trình có hướng** (là DAG).

```text
mặc áo lót → mặc áo sơ mi → thắt cà vạt → mặc vest
             mặc quần    → đi giày
thứ tự hợp lệ: áo lót, quần, sơ mi, giày, cà vạt, vest   (không duy nhất)
```

### Thuật toán Kahn (BFS theo bậc vào)

**Bậc vào** của `v` = số cạnh đi vào `v` = số việc phải làm trước `v`.

```text
q = các đỉnh bậc vào 0
while q: lấy u, ghi u; với mỗi u → v: indeg[v] -= 1, nếu về 0 thì thêm v vào q
nếu ghi được ít hơn n đỉnh → có chu trình
```

In [8]:
from collections import deque

def kahn_demo(n, edges):
    adj = [[] for _ in range(n)]
    indeg = [0] * n
    for u, v in edges:
        adj[u].append(v)
        indeg[v] += 1
    q = deque(i for i in range(n) if indeg[i] == 0)
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for v in adj[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return order if len(order) == n else None

print("DAG:", kahn_demo(5, [(0, 1), (0, 2), (1, 3), (2, 3), (3, 4)]))
print("có chu trình 1→2→3→1:", kahn_demo(4, [(0, 1), (1, 2), (2, 3), (3, 1)]))

DAG: [0, 1, 2, 3, 4]
có chu trình 1→2→3→1: None


### DFS ba màu

- **Trắng**: chưa thăm. **Xám**: đang trên đường đi đệ quy. **Đen**: đã xong.
- Gặp cạnh tới đỉnh **xám** → **chu trình**.
- Ghi đỉnh khi nó chuyển sang **đen** (thứ tự hậu tố), rồi **đảo ngược** → thứ tự tô-pô.

Vì sao đảo ngược? Với cạnh `u → v`: khi `u` xong thì `v` đã xong từ trước (hoặc đã đen sẵn) → `v` đứng trước `u` trong hậu tố → đảo lại thì `u` trước `v`.

In [9]:
def dfs_topo_demo(n, edges):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
    color = [0] * n
    post = []
    def visit(u):
        color[u] = 1
        for v in adj[u]:
            if color[v] == 1:
                return False                   # chu trình
            if color[v] == 0 and not visit(v):
                return False
        color[u] = 2
        post.append(u)
        return True
    for u in range(n):
        if color[u] == 0 and not visit(u):
            return None
    return post[::-1]

print("DAG:", dfs_topo_demo(5, [(0, 1), (0, 2), (1, 3), (2, 3), (3, 4)]))
print("có chu trình:", dfs_topo_demo(4, [(0, 1), (1, 2), (2, 3), (3, 1)]))

DAG: [0, 2, 1, 3, 4]
có chu trình: None


### Độ phức tạp

Cả hai cách: `O(V + E)` thời gian, `O(V + E)` bộ nhớ. Kahn không đệ quy nên an toàn với đồ thị dạng chuỗi dài.

### Lỗi hay gặp

- Chiều cạnh và bậc vào không nhất quán (`u → v` thì bậc vào tăng ở `v`).
- DFS chỉ dùng "đã thăm" (hai màu) → báo chu trình sai khi hai nhánh gặp lại cùng một đỉnh.
- Quên **đảo ngược** thứ tự hậu tố.
- Quên kiểm tra số đỉnh lấy ra ở Kahn → trả về thứ tự thiếu đỉnh khi có chu trình.

## 2.4 Áp dụng vào bài này

Như #207 nhưng phải trả về **thứ tự tô-pô**: thứ tự lấy đỉnh ra ở **Kahn**, hoặc thứ tự **hậu tố** của **DFS** rồi **đảo ngược**. Có chu trình thì trả `[]`. Vì đáp án không duy nhất, kiểm tra bằng hàm xác nhận "mọi cạnh đi từ trước ra sau".

# 3. Python cơ bản cần biết

In [10]:
from typing import List
from collections import deque
import random, time

order = [0, 2, 1, 3]
pos = {v: i for i, v in enumerate(order)}
print("vị trí của mỗi môn trong thứ tự:", pos)
print("đảo ngược danh sách: order[::-1] =", order[::-1])

vị trí của mỗi môn trong thứ tự: {0: 0, 2: 1, 1: 2, 3: 3}
đảo ngược danh sách: order[::-1] = [3, 1, 2, 0]


# 4. Kiểm tra một thứ tự có hợp lệ không

Vì đáp án không duy nhất, ta không thể so sánh `==` với một đáp án cố định. Thay vào đó kiểm tra:

1. Là một **hoán vị** của `0 .. n−1`.
2. Với mọi `[a, b]`: `b` đứng **trước** `a`.

In [11]:
def is_valid_order(num: int, prereq: List[List[int]], order: List[int]) -> bool:
    if sorted(order) != list(range(num)):
        return False
    pos = {v: i for i, v in enumerate(order)}
    return all(pos[b] < pos[a] for a, b in prereq)

ex = [[1, 0], [2, 0], [3, 1], [3, 2]]
print(is_valid_order(4, ex, [0, 1, 2, 3]), is_valid_order(4, ex, [0, 2, 1, 3]), is_valid_order(4, ex, [1, 0, 2, 3]))

True True False


# 5. Cách 1: Kahn — ghi lại thứ tự lấy ra

Giống hệt #207, chỉ thay "đếm `taken`" bằng "**ghi `order.append(u)`**". Cuối cùng: nếu `len(order) == numCourses` thì trả về `order`, ngược lại `[]`.

In [12]:
def find_order_kahn(num: int, prereq: List[List[int]]) -> List[int]:
    graph = [[] for _ in range(num)]
    indeg = [0] * num
    for a, b in prereq:
        graph[b].append(a)
        indeg[a] += 1
    q = deque(i for i in range(num) if indeg[i] == 0)
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for v in graph[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return order if len(order) == num else []

print(find_order_kahn(4, ex), find_order_kahn(2, [[1, 0], [0, 1]]))

[0, 1, 2, 3] []


# 6. Cách 2: DFS — đảo ngược thứ tự hậu tố

DFS ba màu như #207. Khi một đỉnh `u` **tô đen** (mọi đỉnh nó mở khóa đã xong), ghi `u` vào `post`.

**Vì sao `post[::-1]` là thứ tự tô-pô?** Xét cạnh `u → v` (học `u` trước `v`). Lúc DFS xét cạnh này:

- `v` trắng → DFS vào `v`, `v` xong **trước** `u` ⟹ `v` đứng trước `u` trong `post`.
- `v` đen → `v` đã xong từ trước ⟹ cũng đứng trước `u`.
- `v` xám → chu trình (không xảy ra khi đồ thị là DAG).

Vậy mọi cạnh `u → v` đều có `v` trước `u` trong `post` ⟹ **đảo ngược** lại thì `u` trước `v`. ✔

In [13]:
def find_order_dfs(num: int, prereq: List[List[int]]) -> List[int]:
    graph = [[] for _ in range(num)]
    for a, b in prereq:
        graph[b].append(a)
    color = [0] * num
    post = []
    def dfs(u) -> bool:                       # True nếu gặp chu trình
        color[u] = 1
        for v in graph[u]:
            if color[v] == 1 or (color[v] == 0 and dfs(v)):
                return True
        color[u] = 2
        post.append(u)
        return False
    for u in range(num):
        if color[u] == 0 and dfs(u):
            return []
    return post[::-1]

print(find_order_dfs(4, ex), find_order_dfs(2, [[1, 0], [0, 1]]))

[0, 2, 1, 3] []


# 7. Mô phỏng DFS hậu tố

In [14]:
def trace_dfs(num, prereq):
    graph = [[] for _ in range(num)]
    for a, b in prereq:
        graph[b].append(a)
    color = [0] * num
    post = []
    def dfs(u, depth):
        color[u] = 1
        print("   " * depth + f"vào {u} (xám)")
        for v in graph[u]:
            if color[v] == 0:
                dfs(v, depth + 1)
            else:
                print("   " * (depth + 1) + f"{u} → {v}: {v} đã {'đen' if color[v] == 2 else 'xám'}")
        color[u] = 2
        post.append(u)
        print("   " * depth + f"xong {u} (đen) → post = {post}")
    for u in range(num):
        if color[u] == 0:
            dfs(u, 0)
    print("đảo ngược:", post[::-1], "| hợp lệ?", is_valid_order(num, prereq, post[::-1]))

trace_dfs(4, ex)

vào 0 (xám)
   vào 1 (xám)
      vào 3 (xám)
      xong 3 (đen) → post = [3]
   xong 1 (đen) → post = [3, 1]
   vào 2 (xám)
      2 → 3: 3 đã đen
   xong 2 (đen) → post = [3, 1, 2]
xong 0 (đen) → post = [3, 1, 2, 0]
đảo ngược: [0, 2, 1, 3] | hợp lệ? True


# 8. Phiên bản đúng format LeetCode

In [15]:
class Solution:
    def findOrder(self, numCourses: int, prerequisites: List[List[int]]) -> List[int]:
        graph = [[] for _ in range(numCourses)]
        indeg = [0] * numCourses
        for a, b in prerequisites:
            graph[b].append(a)
            indeg[a] += 1
        q = deque(i for i in range(numCourses) if indeg[i] == 0)
        order = []
        while q:
            u = q.popleft()
            order.append(u)
            for v in graph[u]:
                indeg[v] -= 1
                if indeg[v] == 0:
                    q.append(v)
        return order if len(order) == numCourses else []

In [16]:
from itertools import permutations

def check(name, num, prereq, should_exist):
    out = Solution().findOrder(num, prereq)
    ok = is_valid_order(num, prereq, out) if should_exist else out == []
    assert ok, (name, out)
    print(f"{name:<16} PASS  {out}")


def exists_order_brute(num, prereq):
    return any(is_valid_order(num, prereq, list(p)) for p in permutations(range(num)))


check("Ví dụ 1", 2, [[1, 0]], True)
check("Ví dụ 2", 4, ex, True)
check("Ví dụ 3", 1, [], True)
check("Có chu trình", 2, [[1, 0], [0, 1]], False)
check("Chu trình con", 4, [[1, 0], [2, 1], [1, 2], [3, 0]], False)

sol = Solution()
random.seed(210)
for _ in range(1500):
    n = random.randint(1, 6)
    pairs = {(random.randrange(n), random.randrange(n)) for _ in range(random.randint(0, 9))}
    prereq = [[a, b] for a, b in pairs if a != b]
    exists = exists_order_brute(n, prereq)
    for f in (sol.findOrder, find_order_dfs):
        out = f(n, prereq)
        assert (is_valid_order(n, prereq, out) if exists else out == []), (n, prereq, out)
print("1500 đồ thị ngẫu nhiên: Kahn và DFS đều cho thứ tự hợp lệ (đối chiếu duyệt mọi hoán vị)")

n = 2000
dense = [[a, b] for a in range(n) for b in range(max(0, a - 50), a)]       # DAG ~100k cạnh
t0 = time.time(); out = sol.findOrder(n, dense); t1 = time.time()
print(f"2000 môn, {len(dense)} cạnh: hợp lệ = {is_valid_order(n, dense, out)}, {t1 - t0:.3f}s")

Ví dụ 1          PASS  [0, 1]
Ví dụ 2          PASS  [0, 1, 2, 3]
Ví dụ 3          PASS  [0]
Có chu trình     PASS  []
Chu trình con    PASS  []


1500 đồ thị ngẫu nhiên: Kahn và DFS đều cho thứ tự hợp lệ (đối chiếu duyệt mọi hoán vị)
2000 môn, 98725 cạnh: hợp lệ = True, 0.048s


# 9. Vì sao đúng? Độ phức tạp?

**Kahn:** một đỉnh chỉ được thêm vào `order` khi bậc vào về 0, tức **mọi** môn tiên quyết của nó đã nằm trong `order` → mọi cạnh đều đi từ trước ra sau. Nếu có chu trình, các đỉnh trong chu trình không bao giờ được thêm → `len(order) < n` → trả `[]`.

**DFS:** chứng minh ở mục 6.

```text
Time:  O(V + E)
Space: O(V + E)
```

Lưu ý: số cạnh có thể tới `n(n−1)` ≈ 4·10⁶ — vẫn tuyến tính theo kích thước đầu vào.

# 10. Những lỗi dễ mắc

## Lỗi 1: DFS dùng thứ tự hậu tố mà **quên đảo ngược**

In [17]:
def wrong_no_reverse(num, prereq):
    out = find_order_dfs(num, prereq)
    return out[::-1]                           # tức là trả về post nguyên bản

w = wrong_no_reverse(4, ex)
print("Sai:", w, "| hợp lệ?", is_valid_order(4, ex, w))

Sai: [3, 1, 2, 0] | hợp lệ? False


## Lỗi 2: Dùng thứ tự **tiền tố** của DFS (ghi khi vào đỉnh)

Ghi đỉnh lúc vừa vào nghe có vẻ tự nhiên ("học cha trước con"), nhưng một đỉnh có thể có **nhiều** môn tiên quyết, và DFS có thể vào nó trước khi thăm hết các môn tiên quyết kia.

In [18]:
def wrong_preorder(num, prereq):
    graph = [[] for _ in range(num)]
    for a, b in prereq:
        graph[b].append(a)
    seen, pre = [False] * num, []
    def dfs(u):
        seen[u] = True
        pre.append(u)                           # SAI: ghi khi vào
        for v in graph[u]:
            if not seen[v]:
                dfs(v)
    for u in range(num):
        if not seen[u]:
            dfs(u)
    return pre

w = wrong_preorder(4, ex)
print("Kim cương — tiền tố:", w, "| hợp lệ?", is_valid_order(4, ex, w))
w = wrong_preorder(3, [[0, 2], [1, 0]])
print("[[0,2],[1,0]] — tiền tố:", w, "| hợp lệ?", is_valid_order(3, [[0, 2], [1, 0]], w))

Kim cương — tiền tố: [0, 1, 3, 2] | hợp lệ? False
[[0,2],[1,0]] — tiền tố: [0, 1, 2] | hợp lệ? False


Cả hai đều sai. Kim cương: DFS đi `0 → 1 → 3` và ghi `3` **trước** khi thăm `2`, trong khi `3` cần cả `1` lẫn `2`. Ví dụ thứ hai: DFS bắt đầu từ `0` ghi `0` ngay, trong khi `0` cần `2` học trước. Thứ tự hậu tố đảo ngược thì không mắc lỗi này: một đỉnh chỉ được ghi khi đã xong — mọi đỉnh nó mở khóa đều đã ghi trước nó.

## Lỗi 3: Quên kiểm tra chu trình — trả về thứ tự thiếu môn

In [19]:
def wrong_no_check(num, prereq):
    graph = [[] for _ in range(num)]
    indeg = [0] * num
    for a, b in prereq:
        graph[b].append(a); indeg[a] += 1
    q = deque(i for i in range(num) if indeg[i] == 0)
    order = []
    while q:
        u = q.popleft(); order.append(u)
        for v in graph[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return order                                # SAI: không so với num

print("Chu trình con — sai:", wrong_no_check(4, [[1, 0], [2, 1], [1, 2], [3, 0]]), "| đúng: []")

Chu trình con — sai: [0, 3] | đúng: []


## Lỗi 4: So sánh với **một** đáp án cố định khi test

`[0, 2, 1, 3]` và `[0, 1, 2, 3]` đều đúng. Test bằng `==` sẽ báo sai oan → luôn dùng `is_valid_order`.

# 11. Mẹo nhận diện pattern

- **Cần thứ tự thực hiện** thỏa các ràng buộc "trước/sau" → sắp xếp tô-pô (Kahn hoặc DFS hậu tố đảo ngược).
- **Thứ tự tô-pô nhỏ nhất theo từ điển** → Kahn với **heap** thay cho hàng đợi.
- **Chia thành "học kỳ"** (các môn học song song) → Kahn theo **từng tầng** (#1136).
- **Đường đi dài nhất trên DAG** → DP theo thứ tự tô-pô.

Bài liên quan: **#207 Course Schedule**, **#269 Alien Dictionary** (Hard), **#444 Sequence Reconstruction**, **#1136 Parallel Courses**, **#1203 Sort Items by Groups Respecting Dependencies** (Hard), **#2115 Find All Possible Recipes from Given Supplies**.

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `numCourses = 6`, `prerequisites = [[1, 0], [2, 0], [3, 1], [4, 1], [4, 2], [5, 3], [5, 4]]`, chạy Kahn và viết thứ tự nhận được.

## Bài 2 — Điền chỗ trống

In [20]:
def exercise_2_find_order(numCourses: int, prerequisites: List[List[int]]) -> List[int]:
    graph = [[] for _ in range(numCourses)]
    indeg = [0] * numCourses
    for a, b in prerequisites:
        graph[b].append(a)
        indeg[a] += 1
    # TODO
    pass


# assert is_valid_order(4, [[1, 0], [2, 0], [3, 1], [3, 2]], exercise_2_find_order(4, [[1, 0], [2, 0], [3, 1], [3, 2]]))
# assert exercise_2_find_order(2, [[1, 0], [0, 1]]) == []
# assert exercise_2_find_order(1, []) == [0]

## Bài 3 — Biến thể: thứ tự tô-pô **nhỏ nhất theo từ điển**

Gợi ý: Kahn với `heapq` — luôn lấy môn có số hiệu nhỏ nhất trong các môn đã sẵn sàng.

In [21]:
def exercise_3_lex_smallest_order(numCourses: int, prerequisites: List[List[int]]) -> List[int]:
    # TODO
    pass


# assert exercise_3_lex_smallest_order(4, [[1, 0], [2, 0], [3, 1], [3, 2]]) == [0, 1, 2, 3]
# assert exercise_3_lex_smallest_order(4, [[0, 3], [1, 3]]) == [2, 3, 0, 1]
# assert exercise_3_lex_smallest_order(2, [[1, 0], [0, 1]]) == []

## Bài 4 — Biến thể: Parallel Courses (LeetCode #1136)

`relations[i] = [prev, next]` (đánh số từ **1**), mỗi học kỳ học bao nhiêu môn cũng được miễn đủ môn tiên quyết. Trả về số học kỳ tối thiểu, hoặc `-1` nếu không thể.

In [22]:
def exercise_4_minimum_semesters(n: int, relations: List[List[int]]) -> int:
    # TODO
    pass


# assert exercise_4_minimum_semesters(3, [[1, 3], [2, 3]]) == 2
# assert exercise_4_minimum_semesters(3, [[1, 2], [2, 3], [3, 1]]) == -1
# assert exercise_4_minimum_semesters(4, [[1, 2], [2, 3], [3, 4]]) == 4

## Bài 5 — Mini challenge: Alien Dictionary (LeetCode #269, Hard)

Cho danh sách từ **đã sắp** theo bảng chữ cái của người ngoài hành tinh. Trả về một thứ tự chữ cái hợp lệ (chuỗi chứa đủ mọi chữ xuất hiện), hoặc `""` nếu mâu thuẫn. Gợi ý: so hai từ kề nhau, ký tự **khác đầu tiên** cho một cạnh; cẩn thận trường hợp `["abc", "ab"]` (mâu thuẫn).

In [23]:
def exercise_5_alien_order(words: List[str]) -> str:
    # TODO
    pass


def is_valid_alien(words, order):
    if sorted(order) != sorted(set("".join(words))):
        return False
    rank = {c: i for i, c in enumerate(order)}
    keys = [[rank[c] for c in w] for w in words]
    return keys == sorted(keys)

# assert is_valid_alien(["wrt", "wrf", "er", "ett", "rftt"], exercise_5_alien_order(["wrt", "wrf", "er", "ett", "rftt"]))
# assert exercise_5_alien_order(["z", "x", "z"]) == ""
# assert exercise_5_alien_order(["abc", "ab"]) == ""
# assert sorted(exercise_5_alien_order(["z", "x"])) == ["x", "z"] and exercise_5_alien_order(["z", "x"]) == "zx"

# 13. Gợi ý

**Bài 1**
- Mức 1: `Solution().findOrder(...)` sau khi tự làm (nhớ: hàng đợi FIFO).

**Bài 3**
- Mức 1: thay `deque` bằng list + `heapq.heappush/heappop`.

**Bài 4**
- Mức 1: Kahn theo tầng: mỗi vòng `while` lấy **toàn bộ** hàng đợi hiện tại = một học kỳ.
- Mức 2: đếm số môn đã học; nếu `< n` thì `-1`.

**Bài 5**
- Mức 1: đỉnh = mọi ký tự xuất hiện (kể cả ký tự không có cạnh).
- Mức 2: với cặp `(w1, w2)` kề nhau: tìm vị trí khác đầu tiên → cạnh `w1[i] → w2[i]`; nếu không có vị trí khác và `len(w1) > len(w2)` → trả `""`.
- Mức 3: tránh thêm cạnh trùng làm sai bậc vào (dùng `set` cho mỗi đỉnh).

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [24]:
b1 = [[1, 0], [2, 0], [3, 1], [4, 1], [4, 2], [5, 3], [5, 4]]
o1 = Solution().findOrder(6, b1)
print(o1, is_valid_order(6, b1, o1))

[0, 1, 2, 3, 4, 5] True


## Đáp án Bài 2

In [25]:
def answer_2_find_order(numCourses: int, prerequisites: List[List[int]]) -> List[int]:
    return Solution().findOrder(numCourses, prerequisites)

assert is_valid_order(4, [[1, 0], [2, 0], [3, 1], [3, 2]], answer_2_find_order(4, [[1, 0], [2, 0], [3, 1], [3, 2]]))
assert answer_2_find_order(2, [[1, 0], [0, 1]]) == []
assert answer_2_find_order(1, []) == [0]
print("Bài 2 PASS (giống Solution ở mục 8)")

Bài 2 PASS (giống Solution ở mục 8)


## Đáp án Bài 3

In [26]:
import heapq

def answer_3_lex_smallest_order(numCourses: int, prerequisites: List[List[int]]) -> List[int]:
    graph = [[] for _ in range(numCourses)]
    indeg = [0] * numCourses
    for a, b in prerequisites:
        graph[b].append(a)
        indeg[a] += 1
    h = [i for i in range(numCourses) if indeg[i] == 0]
    heapq.heapify(h)
    order = []
    while h:
        u = heapq.heappop(h)
        order.append(u)
        for v in graph[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                heapq.heappush(h, v)
    return order if len(order) == numCourses else []

assert answer_3_lex_smallest_order(4, [[1, 0], [2, 0], [3, 1], [3, 2]]) == [0, 1, 2, 3]
assert answer_3_lex_smallest_order(4, [[0, 3], [1, 3]]) == [2, 3, 0, 1]
assert answer_3_lex_smallest_order(2, [[1, 0], [0, 1]]) == []
for _ in range(800):
    n = random.randint(1, 6)
    pairs = {(random.randrange(n), random.randrange(n)) for _ in range(random.randint(0, 8))}
    prereq = [[a, b] for a, b in pairs if a != b]
    valid = [list(p) for p in permutations(range(n)) if is_valid_order(n, prereq, list(p))]
    assert answer_3_lex_smallest_order(n, prereq) == (min(valid) if valid else [])
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [27]:
def answer_4_minimum_semesters(n: int, relations: List[List[int]]) -> int:
    graph = [[] for _ in range(n + 1)]
    indeg = [0] * (n + 1)
    for p, q in relations:
        graph[p].append(q)
        indeg[q] += 1
    cur = [i for i in range(1, n + 1) if indeg[i] == 0]
    semesters = taken = 0
    while cur:
        semesters += 1
        taken += len(cur)
        nxt = []
        for u in cur:
            for v in graph[u]:
                indeg[v] -= 1
                if indeg[v] == 0:
                    nxt.append(v)
        cur = nxt
    return semesters if taken == n else -1

assert answer_4_minimum_semesters(3, [[1, 3], [2, 3]]) == 2
assert answer_4_minimum_semesters(3, [[1, 2], [2, 3], [3, 1]]) == -1
assert answer_4_minimum_semesters(4, [[1, 2], [2, 3], [3, 4]]) == 4
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [28]:
def answer_5_alien_order(words: List[str]) -> str:
    chars = []
    for w in words:
        for c in w:
            if c not in chars:
                chars.append(c)
    adj = {c: set() for c in chars}
    indeg = {c: 0 for c in chars}
    for w1, w2 in zip(words, words[1:]):
        for a, b in zip(w1, w2):
            if a != b:
                if b not in adj[a]:
                    adj[a].add(b)
                    indeg[b] += 1
                break
        else:
            if len(w1) > len(w2):
                return ""
    q = deque(c for c in chars if indeg[c] == 0)
    out = []
    while q:
        c = q.popleft()
        out.append(c)
        for d in sorted(adj[c]):
            indeg[d] -= 1
            if indeg[d] == 0:
                q.append(d)
    return "".join(out) if len(out) == len(chars) else ""

assert is_valid_alien(["wrt", "wrf", "er", "ett", "rftt"], answer_5_alien_order(["wrt", "wrf", "er", "ett", "rftt"]))
assert answer_5_alien_order(["z", "x", "z"]) == ""
assert answer_5_alien_order(["abc", "ab"]) == ""
assert sorted(answer_5_alien_order(["z", "x"])) == ["x", "z"] and answer_5_alien_order(["z", "x"]) == "zx"
print("thứ tự cho ví dụ 1:", answer_5_alien_order(["wrt", "wrf", "er", "ett", "rftt"]))
for _ in range(1500):
    alpha = random.sample("abcd", random.randint(1, 4))
    rank = {c: i for i, c in enumerate(alpha)}
    words = sorted({"".join(random.choice(alpha) for _ in range(random.randint(1, 3))) for _ in range(random.randint(1, 5))}, key=lambda w: [rank[c] for c in w])
    out = answer_5_alien_order(words)
    assert is_valid_alien(words, out), (words, out)
print("Bài 5 PASS (danh sách từ sinh từ một bảng chữ cái ẩn luôn nhận được thứ tự hợp lệ)")

thứ tự cho ví dụ 1: wertf
Bài 5 PASS (danh sách từ sinh từ một bảng chữ cái ẩn luôn nhận được thứ tự hợp lệ)


# 15. Tổng kết

> Thứ tự tô-pô = **thứ tự lấy ra ở Kahn** (trả `[]` nếu lấy ra chưa đủ `numCourses` đỉnh), hoặc **hậu tố DFS đảo ngược** (trả `[]` nếu gặp đỉnh xám). Đáp án không duy nhất → kiểm tra bằng hàm `is_valid_order`, không so `==`. `O(V + E)`.

---

## Checklist trước khi nộp

- [ ] Cạnh `b → a`, bậc vào tăng ở `a`?
- [ ] So `len(order)` với `numCourses` để phát hiện chu trình?
- [ ] DFS: dùng hậu tố **và** đảo ngược?
- [ ] Test bằng kiểm tra tính hợp lệ, không phải so một đáp án?